# 🔎 Semantic Search Indexer (Kaggle Version)
**Build a neural search index from your CSV data.**

1. **Initialize** environment.
2. **Load Data** (Scans your `/kaggle/input` datasets).
3. **Process** the data to generate embeddings (Using GPU).
4. **Export** (Saves `jav_search_index.zip` to the Output section for download).

In [ ]:
# @title 1. Initialize Environment
# @markdown Installs necessary libraries (LanceDB, Sentence-Transformers).

!pip install -q lancedb sentence-transformers pandas numpy tqdm

import os
import re
import shutil
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
from IPython.display import display, Markdown

# Enable progress bars for pandas
tqdm.pandas()

# Kaggle specific paths
INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

display(Markdown("✅ **Libraries installed.** Ready to process."))

In [ ]:
# @title 2. Load Data
# @markdown This script looks for `final_api_data.csv` (or any CSV) in your attached Kaggle Datasets.
# @markdown **Instruction:** Use the **'Add Data'** button in the sidebar to upload your CSV before running this.

TARGET_FILENAME = 'final_api_data.csv'
found_file_path = None

print("🔍 Scanning /kaggle/input for CSV files...")

# Walk through input directory to find the file
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        # Priority 1: Exact match
        if file == TARGET_FILENAME:
            found_file_path = os.path.join(root, file)
            break
        # Priority 2: Any CSV (if exact match not found yet)
        elif file.endswith(".csv") and found_file_path is None:
            found_file_path = os.path.join(root, file)
    if found_file_path and found_file_path.endswith(TARGET_FILENAME):
        break

if found_file_path:
    # We copy it to working dir to ensure read/write permissions are standard
    shutil.copy(found_file_path, os.path.join(WORKING_DIR, TARGET_FILENAME))
    display(Markdown(f"✅ **Data Loaded!** Found `{os.path.basename(found_file_path)}` copied to workspace."))
else:
    display(Markdown("❌ **Error:** No CSV file found in `/kaggle/input`. Please click 'Add Data' and upload your CSV."))

In [ ]:
# @title 3. Process & Index (Stream Optimized)
# @markdown **Features:** 2x T4 Parallelism + RAM Protection (Stream Loading) + Fixed Indexing

import lancedb
import pandas as pd
import numpy as np
import os
import re
import shutil
import torch
import gc
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

# --- CONFIG ---
MODEL_NAME = "microsoft/harrier-oss-v1-0.6b"
CSV_FILE = os.path.join(WORKING_DIR, "final_api_data.csv")
DB_FOLDER = os.path.join(WORKING_DIR, "jav_search_index")
TABLE_NAME = "videos"

# Optimized for visibility and speed
CHUNK_SIZE = 5000

# --- HELPER FUNCTIONS ---
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"\.(mp4|wmv|avi|mkv|iso)", "", text)
    text = re.sub(r"\[.*?\]", " ", text)
    text = re.sub(r"\(.*?\)", " ", text)
    noise = ["fhd", "hd", "sd", "1080p", "4k", "vr", "uncensored", "leaked"]
    pattern = r"\b(" + "|".join(noise) + r")\b"
    text = re.sub(pattern, "", text)
    return re.sub(r"\s+", " ", text).strip()

def create_rich_context(row):
    title = clean_text(str(row.get("title", "")))
    jp_title = clean_text(str(row.get("jptitle", "")))
    dvd_id = str(row.get("dvdid", "")).strip()
    actresses = str(row.get("actress_names", "")).replace(",", " ")
    
    text_parts = []
    if actresses and actresses.lower() != "nan":
        text_parts.append(f"Starring: {actresses}.")
    if title: 
        text_parts.append(title)
    if jp_title and jp_title.lower() != "nan" and jp_title != title: 
        text_parts.append(jp_title)
    if dvd_id: 
        text_parts.append(dvd_id)
    
    return " ".join(text_parts)

# --- EXECUTION ---
if os.path.exists(DB_FOLDER):
    shutil.rmtree(DB_FOLDER) 
os.makedirs(DB_FOLDER, exist_ok=True)

if not os.path.exists(CSV_FILE):
    print("❌ CSV File not found! Ensure Step 2 ran successfully.")
else:
    db = lancedb.connect(DB_FOLDER)

    print(f"🧠 Loading Model: {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME)
    
    # Setup Multi-GPU Pool
    if torch.cuda.device_count() > 1:
        print(f"⚡⚡ MULTI-GPU ACTIVATED: Found {torch.cuda.device_count()} GPUs!")
        pool = model.start_multi_process_pool()
        USE_MULTI = True
    else:
        print("⚠️ Only 1 GPU found. Running standard mode.")
        model.to("cuda")
        USE_MULTI = False

    # Get total lines
    with open(CSV_FILE, "r", encoding="utf-8", errors="ignore") as f:
        total_rows = sum(1 for line in f) - 1
    print(f"Total Rows: {total_rows}")

    print("🚀 Starting Stream Processing...")
    
    csv_iterator = pd.read_csv(CSV_FILE, dtype=str, chunksize=CHUNK_SIZE)
    first_batch = True
    pbar = tqdm(total=total_rows, desc="Indexing Rows")

    for i, batch_df in enumerate(csv_iterator):
        batch_df = batch_df.fillna("")
        batch_df["search_text"] = batch_df.apply(create_rich_context, axis=1)
        batch_df = batch_df[batch_df["search_text"].str.len() > 5]
        
        if batch_df.empty:
            pbar.update(CHUNK_SIZE)
            continue

        sentences = batch_df["search_text"].tolist()

        if USE_MULTI:
            embeddings = model.encode(sentences, pool=pool, normalize_embeddings=True)
        else:
            embeddings = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False, batch_size=64)

        if isinstance(embeddings, torch.Tensor):
            embeddings = embeddings.cpu().numpy()

        chunk_data = []
        records = batch_df.to_dict("records")
        
        for idx, row in enumerate(records):
            chunk_data.append({
                "vector": embeddings[idx],
                "dvdid": str(row.get("dvdid", "")),
                "title": str(row.get("title", "")),
                "jptitle": str(row.get("jptitle", "")),
                "actress_names": str(row.get("actress_names", "")),
                "releasedate": str(row.get("releasedate", "")),
                "image": str(row.get("image", "")),
                "generated_url": str(row.get("generated_url", ""))
            })

        if first_batch:
            table = db.create_table(TABLE_NAME, data=chunk_data, mode="overwrite")
            first_batch = False
        else:
            table.add(chunk_data)

        pbar.update(len(batch_df))
        del batch_df, sentences, embeddings, chunk_data, records
        gc.collect()

    pbar.close()

    if USE_MULTI:
        model.stop_multi_process_pool(pool)

    print(f"✅ Indexing complete. Final DB size: {len(table)} rows.")

    print("⚙️ Building optimized index (IVF-PQ)...")
    
    # FIXED: num_sub_vectors changed to 64 (1024 / 64 = 16)
    # Reduced num_partitions slightly to 2048 for better fit with ~500k rows
    table.create_index(
        metric="cosine", 
        vector_column_name="vector", 
        num_partitions=2048, 
        num_sub_vectors=64
    )
    print("✅ Optimization Complete.")

In [ ]:
# @title 4. Compress & Export
# @markdown The zip file will be saved to the **Output** section of this Kaggle notebook.

import zipfile

SOURCE_FOLDER = os.path.join(WORKING_DIR, "jav_search_index")
OUTPUT_FILENAME = os.path.join(WORKING_DIR, "jav_search_index.zip")

def zipdir_with_progress(path, ziph):
    total_files = sum([len(files) for r, d, files in os.walk(path)])
    with tqdm(total=total_files, unit="file", desc="📦 Zipping") as pbar:
        for root, dirs, files in os.walk(path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, os.path.join(path, '..'))
                ziph.write(file_path, arcname)
                pbar.update(1)

# --- EXECUTION ---
if os.path.exists(SOURCE_FOLDER):
    print(f"🚀 Zipping database to Output directory...")
    with zipfile.ZipFile(OUTPUT_FILENAME, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipdir_with_progress(SOURCE_FOLDER, zipf)
    
    size_mb = os.path.getsize(OUTPUT_FILENAME) / (1024 * 1024)
    
    display(Markdown(f"## ✅ **SUCCESS!**"))
    display(Markdown(f"**File:** `jav_search_index.zip` ({size_mb:.2f} MB)"))
    display(Markdown("**How to Download:** Go to the **'Output'** tab (right sidebar) or finish the Commit/Save version, then browse the output files to download."))
else:
    print(f"❌ Error: Folder '{SOURCE_FOLDER}' not found.")